# Generation, oversampling, and traceability

This notebook walks through MIMIC generation modes in order: `0` identity, `1` direct, `2` factorised, and `3` joint. Each mode generates minority-class rows for the same two-spiral dataset and shows the resulting class balance and oversampling geometry.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src" / "mimic").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
if NOTEBOOK_DIR.exists() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from mimic import NearestNeighborPrivacyFilter
from mimic_notebook_utils import class_balance, make_two_spiral_frame, run_generation_mode_demo

RANDOM_STATE = 42
MAJORITY_SAMPLES = 300
MINORITY_SAMPLES = MAJORITY_SAMPLES // 3
CAPACITY = 0.25
CONDITION = {"label": "minority"}

df = make_two_spiral_frame(
    majority_samples=MAJORITY_SAMPLES,
    minority_samples=MINORITY_SAMPLES,
    random_state=RANDOM_STATE,
)
minority_needed = df["label"].value_counts()["majority"] - df["label"].value_counts()["minority"]

display(class_balance(df).style.set_caption("Original class balance"))
display(df.head(8).style.set_caption("Original two-spiral training rows"))


## Mode 0: identity

Identity mode generates in the preprocessed feature space and is a transparent baseline.

In [ ]:
mode0 = run_generation_mode_demo(
    df,
    mode=0,
    capacity=0.0,
    n_samples=minority_needed,
    condition=CONDITION,
    random_state=RANDOM_STATE,
)

display(mode0["summary"].style.set_caption("Mode 0 summary"))
display(mode0["balance"].style.set_caption("Mode 0 class balance after oversampling"))
mode0["fig"].suptitle("Mode 0: identity oversampling", y=1.02)


## Mode 1: direct

Direct mode uses learned neural embeddings and deterministic decoding.

In [ ]:
mode1 = run_generation_mode_demo(
    df,
    mode=1,
    capacity=CAPACITY,
    n_samples=minority_needed,
    condition=CONDITION,
    random_state=RANDOM_STATE,
)

display(mode1["summary"].style.set_caption("Mode 1 summary"))
display(mode1["balance"].style.set_caption("Mode 1 class balance after oversampling"))
mode1["fig"].suptitle("Mode 1: direct oversampling", y=1.02)


## Mode 2: factorised

Factorised mode adds stochastic feature-wise conditional decoding.

In [ ]:
mode2 = run_generation_mode_demo(
    df,
    mode=2,
    capacity=CAPACITY,
    n_samples=minority_needed,
    condition=CONDITION,
    random_state=RANDOM_STATE,
)

display(mode2["summary"].style.set_caption("Mode 2 summary"))
display(mode2["balance"].style.set_caption("Mode 2 class balance after oversampling"))
mode2["fig"].suptitle("Mode 2: factorised oversampling", y=1.02)


## Mode 3: joint

Joint mode uses neural conditional evidence and a deterministic joint row decoder.

In [ ]:
mode3 = run_generation_mode_demo(
    df,
    mode=3,
    capacity=CAPACITY,
    n_samples=minority_needed,
    condition=CONDITION,
    random_state=RANDOM_STATE,
)

display(mode3["summary"].style.set_caption("Mode 3 summary"))
display(mode3["balance"].style.set_caption("Mode 3 class balance after oversampling"))
mode3["fig"].suptitle("Mode 3: joint oversampling", y=1.02)


## Mode 3 with nearest-neighbor ambiguity filtering

This repeats joint generation with nearest-neighbor ambiguity filtering enabled. The filter keeps generated embeddings that have nearby non-source neighbours in embedding space.

In [ ]:
mode3_private = run_generation_mode_demo(
    df,
    mode=3,
    capacity=CAPACITY,
    n_samples=minority_needed,
    condition=CONDITION,
    random_state=RANDOM_STATE,
    privacy_filter=NearestNeighborPrivacyFilter(k=5, min_ambiguous_neighbors=2, exclude_generation_sources=False),
)

display(mode3_private["summary"].style.set_caption("Mode 3 privacy-aware summary"))
display(mode3_private["balance"].style.set_caption("Mode 3 privacy-aware class balance after oversampling"))
mode3_private["fig"].suptitle("Mode 3: privacy-aware joint oversampling", y=1.02)
